# SLD Data Preparation

**Selections, jet clustering, and OmniLearn point-cloud inputs.**

This is **notebook 1 of 3** in the SLD reproduction pipeline:

| Notebook | Purpose | Outputs |
|---|---|---|
| **01 – Data preparation** *(this one)* | Apply published selections; cluster jets; build ML inputs | `sld_{hadronic,leptonic,ee,mumu,tautau}.parquet`, `omnilearned_input_*.h5` |
| 02 – Validation plots | Kinematic sanity checks on the selected samples | PDFs in `analysis/plots/` |
| 03 – Asymmetry measurements | Reproduce $A_{LR}$ and $A_\ell$ and combine into $\sin^2\theta_W^{\mathrm{eff}}$ | JSON results, forest plot |

The three notebooks share data through the parquet files produced here.

### Reference analyses

This notebook applies two published SLD selections, both with the year-range covered by the released dataset (1996–1998):

| Channel | Reference | Used for |
|---|---|---|
| Hadronic $Z\to q\bar q$ | [hep-ex/0004026](https://arxiv.org/abs/hep-ex/0004026) — *2000 high-precision $A_{LR}$* | $A_{LR}$ (notebook 03), validation plots (notebook 02), OmniLearn inputs (this notebook) |
| Leptonic $Z\to\ell^+\ell^-$ ($ee$, $\mu\mu$, $\tau\tau$) | [hep-ex/0010015](https://arxiv.org/abs/hep-ex/0010015) — *2001 leptonic-coupling asymmetries* | $A_e$, $A_\mu$, $A_\tau$ (notebook 03) |

### Dataset format

Each event is a record of "bank families" from the SLD reconstruction. This notebook uses only the banks listed in `REQUESTED_BANKS` below; the full set is preserved in the public release.

## 1. Imports and configuration

The root directory for the dataset is controlled by the `SLD_BASE`
environment variable; if it is unset we default to `./sld` relative to the
notebook's working directory. Set it once before launching Jupyter to point
at wherever you placed the released parquet files:

```bash
export SLD_BASE=/path/to/your/SLD
```

In [ ]:
from __future__ import annotations

import glob
import os
from pathlib import Path
from typing import Iterable

import awkward as ak
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Public API of the sld_resurrect package: kinematics provides observable
# builders, selector defines the cut framework (EventSelector.from_preset
# looks up the published presets by name), and the datasets sub-package
# builds the OmniLearn point-cloud inputs.
from sld_resurrect.kinematics import build_particles
from sld_resurrect.selector import EventSelector

SLD_BASE = Path(os.environ.get("SLD_BASE", "./sld"))
DATADIR: str    = str(SLD_BASE / "datasets/minidst_translated/parquet")
OUTPUT_DIR: str = str(SLD_BASE / "datasets/minidst_processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"SLD_BASE   = {SLD_BASE.resolve()}")
print(f"DATADIR    = {DATADIR}")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")

## 2. Load the released parquet shards

The parquet files on disk are the output of the `jazelle` reader applied to the
SLD MiniDST binaries. Each file is a flat collection of `awkward` records keyed
by bank family. We request only the banks this pipeline needs.

In [5]:
import jazelle

REQUESTED_BANKS: list[str] = [
    "IEVENTH",  # event header: run, event, trigger, timestamp
    "PHBM",     # beam info: ecm, per-event polarisation
    "PHPSUM",   # inclusive reconstructed particles
    "PHCHRG",   # raw charged tracks
    "PHKLUS",   # calorimeter clusters (LAC)
    "PHPOINT",  # PHPSUM -> (track, cluster) pointer bank
    "PHWIC",    # muon iron-calorimeter info
]


def load_parquet_dataset(
    data_dir: str = DATADIR,
    pattern: str = "*nrec*.parquet",
    columns: Iterable[str] = REQUESTED_BANKS,
    max_files: int | None = None,
) -> ak.Array:
    """Concatenate every parquet shard in ``data_dir`` into one awkward array.

    Restricting ``columns`` to the bank families actually used speeds up
    loading significantly and keeps peak memory low.
    """
    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if max_files is not None:
        files = files[:max_files]
    if not files:
        raise FileNotFoundError(
            f"No parquet files matching {pattern!r} found in {data_dir!r}"
        )

    arrays = [
        jazelle.from_parquet(path, columns=list(columns))
        for path in tqdm(files, desc="Reading parquet files")
    ]
    return ak.concatenate(arrays)


data = load_parquet_dataset()
print(f"Loaded {len(data):,} events from {DATADIR}")

Reading parquet files:   0%|          | 0/68 [00:00<?, ?it/s]

Loaded 658,241 events


## 3. Build particle 4-momenta

`PHPSUM` is the inclusive particle list of each event. To turn it into a
Lorentz-vector collection we need a mass assumption per entry. The standard
SLD hadronic convention assigns the charged-pion mass ($m_\pi \approx 0.140$
GeV/$c^2$) to every charged track; neutral entries (photons, $K_L^0$, neutrons,
…) are treated as massless. The reconstructed origin point of each particle is
attached as the extra fields `vx`, `vy`, `vz`, which several SLD track-quality
cuts use.

In [6]:
data.PHPSUM

<Array [[{id: 1, px: -0.144, ...}, ...], ...] type='658241 * var * {id: int...'>

In [7]:
particles = build_particles(data)
print(f"Built particle 4-vectors for {len(particles):,} events")
print(f"  mean # particles / event         : "
      f"{ak.mean(ak.num(particles, axis=-1)):.1f}")
print(f"  mean # charged particles / event : "
      f"{ak.mean(ak.num(particles[particles.charge != 0], axis=-1)):.1f}")

Built particle 4-vectors for 658,241 events
  mean # particles / event         : 54.4
  mean # charged particles / event : 13.4


## 4. Published selections used here

We apply five selections: the **2000 hadronic $A_{LR}$** preset and the
**2001 leptonic** preset in its three flavours ($ee$, $\mu\mu$, $\tau\tau$),
plus the channel-agnostic leptonic preselection (for diagnostic use).

### 4.1 Hadronic preset — 2000 $A_{LR}$ ([hep-ex/0004026](https://arxiv.org/abs/hep-ex/0004026))

Calorimeter-led selection: the $A_{LR}$ systematic budget is dominated by
polarimetry, not tracking, so a robust topology + LAC-energy filter is preferred
over event-shape cuts.

* **Track quality** — $p_T > 100$ MeV, angle $> 30^\circ$ from the beam, impact
  parameter within 5 cm radially / 10 cm along the beam direction.
* **Topology** — $\ge 2$ tracks per beam-hemisphere **or** $\ge 4$ in one
  hemisphere. Suppresses Bhabha events, which are forward/backward-peaked and
  rarely populate both sides.
* **Energy** — $E_{\mathrm{LAC}} \ge 22$ GeV; normalised energy imbalance $< 0.6$.

### 4.2 Leptonic preselection — 2001 ([hep-ex/0010015](https://arxiv.org/abs/hep-ex/0010015))

Common to all three dilepton channels:

* **2 – 8 charged tracks** within 1 cm of the interaction point. Removes
  high-multiplicity hadronic events ($\langle n_{\mathrm{ch}}\rangle \approx 20$)
  and beam-related backgrounds.
* **Hemisphere net charges $\{+1, -1\}$** wrt the thrust axis. Ensures
  unambiguous assignment of the forward / backward direction.
* **Year-dependent thrust-axis fiducial** — $|\cos\theta_T| < 0.8$ for 1996,
  $< 0.9$ for 1997–98. Tracks the central-tracker acceptance growth.

### 4.3 Channel-specific cuts (on top of the preselection)

| Channel | Discriminator(s) | Why |
|---|---|---|
| `leptonic_ee` | Per-hemisphere sum of leading-track LAC energies $> 45$ GeV | Electron pairs deposit nearly all $\sqrt s$ in the LAC |
| `leptonic_mumu` | Charged-track invariant mass $> 70$ GeV; max per-hemisphere leading-track LAC $< 14$ GeV | Muons leave $\sim 2$ GeV in the LAC; the mass cut removes $\tau\tau$ and two-photon |
| `leptonic_tautau` | Charged-track mass $< 70$ GeV; $\cos\theta$-dependent LAC veto vs $ee$; hemisphere opening angle $> 160^\circ$; leading-track $p > 4$ GeV; max thrust-hemisphere mass $< 1.6$ GeV | $\tau$ neutrinos carry away energy, so visible mass is depressed |

## 5. Build the selectors and print cutflows

In [ ]:
def show_preset(preset: str) -> EventSelector:
    """Build a selector for ``preset``, print its cutflow, and return it."""
    print(f"\n=== {preset} ===")
    selector = EventSelector.from_preset(preset, data, particles)
    selector.print_cutflow()
    return selector


hadronic_selector = show_preset("hadronic_default")    # 2000 A_LR preset
leptonic_selector = show_preset("leptonic_default")    # 2001 leptonic preselection
ee_selector       = show_preset("leptonic_ee")
mumu_selector     = show_preset("leptonic_mumu")
tautau_selector   = show_preset("leptonic_tautau")

## 6. Save the selected events as parquet

Each selection is applied to the full dataset and the selected slice is written
to disk for the downstream notebooks. Saving the slice (rather than just the
boolean mask) makes notebooks 02 and 03 independent of the raw shards and
removes the need to rebuild particles on every run.

In [ ]:
def apply_and_save(
    selector: EventSelector,
    data: ak.Array,
    label: str,
    output_dir: str = OUTPUT_DIR,
) -> ak.Array:
    """Apply a selection, save the selected event record to parquet, and return it.

    The full event record (all banks) is saved so downstream notebooks can
    rebuild particles, recompute observables, or inspect raw banks without
    needing the original parquet shards.
    """
    mask = selector.mask()
    selected = data[mask]
    n_sel, n_tot = int(mask.sum()), len(data)
    print(f"{label}: {n_sel:,} / {n_tot:,} events ({100 * n_sel / n_tot:.1f}%)")

    out_path = os.path.join(output_dir, f"sld_{label}.parquet")
    ak.to_parquet(selected, out_path)
    print(f"  -> {out_path}")
    return selected


data_hadronic = apply_and_save(hadronic_selector, data, "hadronic")
data_leptonic = apply_and_save(leptonic_selector, data, "leptonic")
data_ee       = apply_and_save(ee_selector,       data, "ee")
data_mumu     = apply_and_save(mumu_selector,     data, "mumu")
data_tautau   = apply_and_save(tautau_selector,   data, "tautau")

## 7. Durham jet clustering on the hadronic sample

At $e^+e^-$ colliders the standard jet algorithm is the **Durham** ($ee\,k_T$)
algorithm, which defines the distance between particles $i$ and $j$ as

$$ y_{ij} \;=\; \frac{2\,\min(E_i^2,\,E_j^2)\,(1 - \cos\theta_{ij})}{E_{\mathrm{vis}}^2}. $$

The measure is infrared- and collinear-safe and adapted to the spherical
geometry of $e^+e^-$ events. We cluster each hadronic event into **exactly
two** exclusive jets, sorted by descending $p_T$. The resulting constituents
feed both the OmniLearn inputs (§8) and the validation $m_Z$ plot in
notebook 02.

In [ ]:
import fastjet
from fastjet._pyjet import AwkwardClusterSequence
from fastjet._swig import JetDefinition


def cluster_exclusive_jets(
    particles: ak.Array,
    n_jets: int = 2,
) -> tuple[ak.Array, ak.Array]:
    """Cluster particles into exactly ``n_jets`` exclusive jets.

    Returns
    -------
    jets : ak.Array
        Jet 4-vectors, shape ``(n_events, n_jets)``, sorted by descending pT.
    constituents : ak.Array
        Constituent particles aligned with ``jets``.
    """
    jet_def = JetDefinition(fastjet.ee_kt_algorithm)
    cluster_seq = AwkwardClusterSequence(particles, jet_def)
    constituents = cluster_seq.exclusive_jets_constituents(n_jets)
    jets = ak.sum(constituents, axis=2)

    pt_order = ak.argsort(jets.pt, axis=1, ascending=False)
    return jets[pt_order], constituents[pt_order]


had_particles = build_particles(data_hadronic)
had_jets, had_constituents = cluster_exclusive_jets(had_particles, n_jets=2)
print(f"Clustered {len(had_jets):,} hadronic events into 2 Durham jets each.")

## 8. OmniLearn point-cloud preparation

To feed SLD events into the OmniLearn foundation model we need fixed-size,
axis-aligned point clouds. Three coordinate-mapping strategies are implemented
in `sld_resurrect.datasets`:

| Strategy | Reference axis | Geometry |
|---|---|---|
| `superjet` | Thrust axis (one per event) | The whole event is one wide-angle jet |
| `hemisphere` | Each Durham jet's own axis | Two embeddings per event, one per hemisphere — matches the angular scale OmniLearn was pre-trained on |
| `boosted_frame` | $+\hat z$ after a per-event rigid rotation | Rotate the event so its thrust axis is along $+z$, then embed in $(\Delta\eta, \Delta\phi)$ |

Per-particle features are $(\Delta\eta,\ \Delta\phi,\ \log p_T,\ \log E)$;
particles are $p_T$-sorted and zero-padded (or clipped) to `max_particles`
entries per event.

In [ ]:
import h5py

from sld_resurrect.datasets.sld import save_strategy_outputs

written = save_strategy_outputs(
    constituents=had_constituents,
    particles=had_particles,
    output_dir=OUTPUT_DIR,
    strategies=("superjet", "hemisphere", "boosted_frame"),
    max_particles=128,
)

for label, path in written.items():
    with h5py.File(path, "r") as f:
        arr = f["data"]
        print(f"  {label:25s} -> {path}  shape={arr.shape}, dtype={arr.dtype}")

## 9. Outputs and next steps

This notebook produced, in `OUTPUT_DIR`:

* `sld_hadronic.parquet`, `sld_leptonic.parquet`, `sld_ee.parquet`,
  `sld_mumu.parquet`, `sld_tautau.parquet` — the selected event records.
* `omnilearned_input_sld_{superjet,hemisphere_leading,hemisphere_subleading,boosted_frame}.h5` —
  ML-ready point clouds.

Continue with:

* **Notebook 02 — Validation plots** for the kinematic sanity checks
  ($m_Z$, event shapes, polarised $\cos\theta$).
* **Notebook 03 — Asymmetry measurements** to extract $A_{LR}$ and $A_\ell$
  from the saved samples.